# Análise Exploratória do Dataset
### Classificação de Faixa de Preço de Celulares — KNN
**Disciplinas:** Matemática para Computação · Inteligência Artificial · Fundamentos de Programação · Fundamentos de Lógica

**UniFAJ**

## Importação das Bibliotecas

In [ ]:
# ==========================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# ==========================================================

# pandas -> leitura e manipulação do dataset
import pandas as pd

# matplotlib -> geração de gráficos
import matplotlib.pyplot as plt

# seaborn -> visualizações estatísticas
import seaborn as sns

## 1. Visão Geral
O dataset possui informações técnicas de smartphones e tem como objetivo classificar os aparelhos em diferentes faixas de preço por meio da variável alvo `price_range`.

In [ ]:
# ==========================================================
# LEITURA DO DATASET
# ==========================================================

# Carrega o arquivo CSV (dado bruto)
df = pd.read_csv("https://drive.google.com/uc?export=download&id=1wF7amDwgNKIxa9Lc5Dm4OXkiDy_N_CYu")

# Mostra as 5 primeiras linhas
print("Primeiras linhas do dataset:")
df.head()

In [ ]:
# ==========================================================
# INFORMAÇÕES GERAIS DO DATASET
# ==========================================================

print("Quantidade de registros e colunas:", df.shape)
print("\nQuantidade de registros:", df.shape[0])
print("Quantidade de colunas:  ", df.shape[1])
print("\nTipos de dados por coluna:")
print(df.dtypes)

| Característica | Valor |
|----------------|-------|
| Quantidade de registros | 2000 |
| Quantidade de colunas | 21 |
| Tipo do problema | Classificação |
| Variável alvo | price_range |
| Quantidade de classes | 4 |

## 2. Verificação de Valores Ausentes
O dataset não apresenta valores ausentes, dispensando técnicas de imputação ou preenchimento de dados.

In [ ]:
# ==========================================================
# VERIFICAÇÃO DE VALORES AUSENTES
# ==========================================================

valores_ausentes = df.isnull().sum()

print("Valores ausentes por coluna:")
print(valores_ausentes)

print("\nTotal de valores ausentes:", valores_ausentes.sum())

## 3. Verificação de Dados Duplicados
Zero registros duplicados foram encontrados no dataset.

In [ ]:
# ==========================================================
# VERIFICAÇÃO DE DADOS DUPLICADOS
# ==========================================================

duplicados = df.duplicated().sum()

print("Quantidade de registros duplicados:", duplicados)

## 4. Distribuição das Classes
O dataset apresenta classes perfeitamente balanceadas, fator importante para evitar viés durante o treinamento do modelo.

In [ ]:
# ==========================================================
# DISTRIBUIÇÃO DAS CLASSES
# ==========================================================

contagem = df["price_range"].value_counts().sort_index()

print("Contagem por classe:")
for classe, qtd in contagem.items():
    labels = {0: "Baixo custo", 1: "Custo médio", 2: "Alto custo", 3: "Custo muito alto"}
    print(f"  Classe {classe} ({labels[classe]}): {qtd} registros")

# Gráfico de distribuição
plt.figure(figsize=(7, 4))
cores = ["#4C9BE8", "#5BBF8A", "#F0A84B", "#E86B5F"]
plt.bar(contagem.index, contagem.values, color=cores, edgecolor="white", width=0.6)
plt.title("Distribuição das Classes — price_range", fontsize=13)
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.xticks([0, 1, 2, 3], ["0 - Baixo", "1 - Médio", "2 - Alto", "3 - Muito alto"])
plt.tight_layout()
plt.show()

## 5. Análise das Variáveis
O dataset contém atributos relacionados às características dos smartphones, como:
- `battery_power`: capacidade da bateria
- `bluetooth`: conectividade Bluetooth
- `clock_speed`: velocidade do processador
- `int_memory`: memória interna
- `ram`: memória RAM
- `px_height` e `px_width`: resolução da tela
- `n_cores`: quantidade de núcleos do processador
- `fc` e `pc`: câmeras frontal e principal

Essas características possuem potencial influência sobre a faixa de preço dos dispositivos.

In [ ]:
# ==========================================================
# ESTATÍSTICAS DESCRITIVAS
# ==========================================================

print("Estatísticas descritivas do dataset:")
df.describe()

## 6. Valores Iguais a Zero
Durante a exploração dos dados, observou-se a presença de valores iguais a zero em algumas colunas.

**Conclusão:** grande parte dos valores iguais a zero representa **ausência de funcionalidades** e não necessariamente erros ou inconsistências nos dados.

In [ ]:
# ==========================================================
# VERIFICAÇÃO DE ZEROS NAS COLUNAS BINÁRIAS
# ==========================================================

# Colunas binárias onde zero significa ausência de funcionalidade
colunas_binarias = ["blue", "four_g", "three_g", "touch_screen", "wifi", "dual_sim"]

print("Quantidade de zeros por coluna binária:")
print("-" * 35)

for col in colunas_binarias:
    zeros   = (df[col] == 0).sum()
    uns     = (df[col] == 1).sum()
    print(f"{col:15} → Sem: {zeros:4}  Com: {uns:4}")

print("\nColunas de câmera com zero (sem câmera):")
print("fc = 0 (sem câmera frontal)   :", (df["fc"] == 0).sum())
print("pc = 0 (sem câmera principal) :", (df["pc"] == 0).sum())

## 7. Curadoria dos Dados
Durante a etapa de preparação dos dados foram realizadas as seguintes ações:

In [ ]:
# ==========================================================
# RENOMEAÇÃO DE COLUNAS
# ==========================================================

# Mapeamento: nome original -> novo nome mais legível
renomear = {
    "blue"      : "bluetooth",
    "fc"        : "front_camera",
    "pc"        : "primary_camera",
    "int_memory": "storage",
    "mobile_wt" : "mobile_weight",
    "sc_h"      : "sc_height",
    "sc_w"      : "sc_width"
}

df = df.rename(columns=renomear)

print("Colunas renomeadas com sucesso!")
print("\nNovos nomes das colunas:")
print(df.columns.tolist())

In [ ]:
# ==========================================================
# REMOÇÃO DA COLUNA m_dep
# ==========================================================

# Removida por apresentar baixa relevância
# para a classificação da faixa de preço

if "m_dep" in df.columns:
    df = df.drop(columns=["m_dep"])
    print("Coluna 'm_dep' removida com sucesso!")
else:
    print("Coluna 'm_dep' não encontrada no dataset.")

print("\nDataset após remoção:")
print("Colunas restantes:", df.shape[1])

## 8. Seleção de Atributos
Após testes com o algoritmo KNN, observou-se que algumas variáveis contribuíam mais para a classificação dos smartphones.

As colunas selecionadas para o modelo final foram: `battery_power`, `clock_speed`, `n_cores` e `ram`.

In [ ]:
# ==========================================================
# SELEÇÃO DE ATRIBUTOS
# ==========================================================

# Atributos selecionados para o modelo final
atributos = ["battery_power", "clock_speed", "n_cores", "ram", "price_range"]

df_final = df[atributos]

print("Atributos selecionados para o modelo:")
for col in atributos[:-1]:
    print(f"  - {col}")
print("  - price_range (variável alvo)")

print("\nDataset final:")
print("Linhas:", df_final.shape[0])
print("Colunas:", df_final.shape[1])

df_final.head()

In [ ]:
# ==========================================================
# CORRELAÇÃO DOS ATRIBUTOS COM A VARIÁVEL ALVO
# ==========================================================

correlacao = df_final.corr()["price_range"].drop("price_range").sort_values(ascending=False)

print("Correlação dos atributos com price_range:")
print(correlacao)

# Gráfico de correlação
plt.figure(figsize=(7, 4))
cores = ["#E86B5F" if v < 0 else "#4C9BE8" for v in correlacao.values]
plt.barh(correlacao.index, correlacao.values, color=cores, edgecolor="white")
plt.title("Correlação dos Atributos com price_range", fontsize=13)
plt.xlabel("Correlação")
plt.axvline(0, color="gray", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.show()

## 9. Conclusão
A análise exploratória demonstrou que o dataset possui **boa qualidade**, sem valores ausentes ou registros duplicados. Além disso, as classes encontram-se **perfeitamente balanceadas**, favorecendo o treinamento de modelos de classificação.

Durante a curadoria dos dados foram realizadas **renomeações** para melhorar a compreensão dos atributos e removida uma variável considerada pouco relevante (`m_dep`). Após testes experimentais, foi realizada uma **seleção de atributos** visando melhorar o desempenho do algoritmo KNN.

Os atributos escolhidos — `battery_power`, `clock_speed`, `n_cores` e `ram` — apresentam maior correlação com a variável alvo `price_range` e são os que serão utilizados na implementação do modelo.